# Improved LLM Plans
Original feature, Only one mental Keggle Mental health information used.
Then, based on information, how should I imporve my model,
In inital Training, gpt model4, and we have serveral information inside.
Next approch, I like to Hugging Datasets and Mental health datasets

Currently, feature we just populate randomly.
Next what, I want is text related llm topic clustering,
Based on token, clustering.


In [25]:
# Hugging Datasets called
raw_data_path = "raw_data"
cleaned_data_path= "cleaned_data"



In [26]:
import pandas as pd
# Inital Raw Data Used.

def clean_keggle_df(data_path):
    df = pd.read_csv(data_path)
    df = df[["questionText", "topics", "re_diagnosis","clean_answer_text"]]
    # Lower case
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    # remove non-world
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    # remove number
    df = df.replace(to_replace=r'\d', value='', regex=True)

    return df

def clean_hugging_df(data_path):
    df = pd.read_csv(data_path)

    df = df[["questionTitle", "questionText", "topic", "answerText"]]
    df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
    df["questionText"] = df["questionTitle"].fillna('') + " " + df["questionText"].fillna('')
    df = df.replace(to_replace=r'[^\w\s]', value="", regex=True)
    df = df.replace(to_replace=r'\d', value='', regex=True)
    df = df[["questionText", "topic", "answerText"]]

    # print(hugging_df.head)
    return df

keggle_df = clean_keggle_df(f"{raw_data_path}/counsel_cleaned.csv")   
print(keggle_df.shape)
keggle_df.to_csv(f"{cleaned_data_path}/cleaned_counsel.csv")
hugging_df = clean_hugging_df(f"{raw_data_path}/huggin_counsel_chat.csv")
print(hugging_df.shape)
hugging_df

(1373, 4)
(2775, 3)


/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_47561/1977113534.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_47561/1977113534.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)


,questionText,topic,answerText
0,do i have too many issues for counseling i hav...,depression,it is very common for people to have multiple ...
1,do i have too many issues for counseling i hav...,depression,ive never heard of someone having too many iss...
2,do i have too many issues for counseling i hav...,depression,absolutely not i strongly recommending workin...
3,do i have too many issues for counseling i hav...,depression,let me start by saying there are never too man...
4,do i have too many issues for counseling i hav...,depression,i just want to acknowledge you for the courage...
...,...,...,...
2770,are some clients more difficult than others wh...,counselingfundamentals,although many clients have the capacity to be ...
2771,are some clients more difficult than others wh...,counselingfundamentals,i usually dont label a client as difficult bec...
2772,are some clients more difficult than others wh...,counselingfundamentals,dang right heh heh and correct me if im wrong...
2773,are some clients more difficult than others wh...,counselingfundamentals,yes just like some relationships outside of ou...


In [27]:
import pandas as pd

# Read File Information
hugging_df = pd.read_csv(f"{cleaned_data_path}/cleaned_hugging.csv")
counsel_df = pd.read_csv(f"{cleaned_data_path}/cleaned_counsel.csv")

# Change Column Name
hugging_df.rename(columns={"questionText": "question_text",
                           "topic": "topics", 
                           "answerText": "answer_text"}, inplace=True)
hugging_df.drop(columns=['Unnamed: 0'], inplace=True)

print("After renaming:", hugging_df.columns)
# Drop unused column
counsel_df.drop(columns=['Unnamed: 0', 're_diagnosis'], inplace=True)
print(counsel_df.columns)
counsel_df.rename(columns={"questionText": "question_text", 
                           "clean_answer_text": "answer_text"}, inplace=True)

# Select target column
hugging_df = hugging_df[["question_text", "topics", "answer_text"]]
counsel_df = counsel_df[["question_text", "topics", "answer_text"]]
print(hugging_df.columns)
print(counsel_df.columns)

# Concat Column
combined_dataset = pd.concat([hugging_df, counsel_df], ignore_index=True)

# Combined Output
print(combined_dataset.columns)
combined_dataset.to_csv(f"{cleaned_data_path}/combined_output.csv")


After renaming: Index(['questionTitle', 'question_text', 'topics', 'answer_text'], dtype='object')
Index(['questionText', 'topics', 'clean_answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')
Index(['question_text', 'topics', 'answer_text'], dtype='object')


In [28]:
# Chat Promt, Design.
combined_dataset = pd.read_csv(f"{cleaned_data_path}/combined_output.csv")
print(combined_dataset.columns)



Index(['Unnamed: 0', 'question_text', 'topics', 'answer_text'], dtype='object')


In [29]:
# NLTK test
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
text = "This is an example. Here is another sentence."
sentences = sent_tokenize(text)

print(sentences)


['This is an example.', 'Here is another sentence.']


[nltk_data] Downloading package punkt_tab to /Users/yoon/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [30]:
# Cleaned Combined Datasets too shorts and too long
import re
from nltk.tokenize import word_tokenize

def is_noisy(text: str) -> bool:
    if re.search(r'[가-힣A-Za-z]', text) is None:
        return True
    cleaned = re.sub(r'[^\w\s]', '', text) 
    if len(cleaned) == 0 or len(cleaned) < len(text) * 0.02:  
        return True
    return False

def clean_combined_dataset(df):
    def token_count(text):
        tokens = sent_tokenize(text)
        return len(tokens)
    # Apply the token_count function to calculate the number of tokens in questions and answers
    df = df.dropna()
    df['q_token_count'] = df['question_text'].apply(token_count)
    df['a_token_count'] = df['answer_text'].apply(token_count)
    
    return df
# print(combined_dataset.columns)
print(f"Before: {combined_dataset.shape}")
cleaned_combined_df = clean_combined_dataset(combined_dataset)
cleaned_combined_df.to_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {cleaned_combined_df.shape}")


Before: (4148, 4)
After: (3985, 6)


/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_47561/339852553.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['q_token_count'] = df['question_text'].apply(token_count)
/var/folders/hy/344rq3v160s1tb_mmkdw_fqc0000gn/T/ipykernel_47561/339852553.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['a_token_count'] = df['answer_text'].apply(token_count)


In [37]:
# Lamma testing Device
from datasets import Dataset, DatasetDict


target_df = pd.read_csv(f"{cleaned_data_path}/cleaned_combined.csv")
print(f"After: {target_df.shape}")
print(f"Columns: {target_df.columns}")

system_prompt = (
    "This is a Mental Health ChatBot assistant designed based on actual consultation data and implemented using real test cases." 
    "Its purpose is to accurately understand users' questions and respond with comforting and helpful messages."
    "The focus is primarily on the question_text, and when necessary, it can provide various empathetic expressions. "
    "In cases where the user's question is unclear or emotionally unstable, the assistant should request additional clarification, while also offering basic empathy and supportive advice"
)

def format_to_chat_messages(example):
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": example["question_text"] + f" based on {example["topics"]}"},
            {"role": "assistant", "content": example["answer_text"]},
        ]
    }

# Grab only nessary columns only
target_df = target_df[["question_text", "topics", "answer_text"]].dropna()
hf_dataset = Dataset.from_pandas(target_df)

# Hugging Face Datasets
hf_dataset = hf_dataset.map(format_to_chat_messages)
hf_dataset = hf_dataset.remove_columns([col for col in hf_dataset.column_names if col != "messages"])
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)

train_data_path = "train_data"
hf_dataset["train"].to_json(f"{train_data_path}/train_dataset.json", orient="records", force_ascii=False)
hf_dataset["test"].to_json(f"{train_data_path}/test_dataset.json", orient="records", force_ascii=False)
print(hf_dataset)

After: (3985, 7)
Columns: Index(['Unnamed: 0.1', 'Unnamed: 0', 'question_text', 'topics', 'answer_text',
       'q_token_count', 'a_token_count'],
      dtype='object')


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 151.97ba/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 3586
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 399
    })
})


In [39]:
### 3.5.4. Llama3 모델 파라미터 설정 
import logging
from dataclasses import dataclass, field
import os
import random
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, TrainingArguments
from trl.commands.cli_utils import  TrlParser
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
        set_seed,

)
from trl import setup_chat_format
from peft import LoraConfig

from trl import (
   SFTTrainer)


from sklearn.model_selection import train_test_split

@dataclass
class ScriptArguments:
    dataset_path: str = field(
        default=None,
        metadata={
            "help": "file path"
        },
    )
    model_name: str = field(
    default=None, metadata={"help": "model id"}
    )
    max_seq_length: int = field(
        default=512, metadata={"help": "data sequnce length for SFT Trainer"}
    )
    question_key: str = field(
    default=None, metadata={"help": "question key"}
    )
    answer_key: str = field(
    default=None, metadata={"help": "answer key"}
    )    


def training_function(script_args, training_args):    
    # load datasets
    train_dataset = load_dataset(
        "json",
        data_files=os.path.join(script_args.dataset_path, "train_dataset.json"),
        split="train",
    )
    test_dataset = load_dataset(
        "json",
        data_files=os.path.join(script_args.dataset_path, "test_dataset.json"),
        split="train",
    )

    # 토크나이저 및 데이터셋 chat_template으로 변경하기      
    tokenizer = AutoTokenizer.from_pretrained(script_args.model_name, use_fast=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.chat_template = LLAMA_3_CHAT_TEMPLATE
    tokenizer.padding_side = 'right'
    
    def template_dataset(examples):
        return{"text":  tokenizer.apply_chat_template(examples["messages"], tokenize=False)}
    
    train_dataset = train_dataset.map(template_dataset, remove_columns=["messages"])
    test_dataset = test_dataset.map(template_dataset, remove_columns=["messages"])
    
    # 데이터가 변화되었는지 확인하기 위해 2개만 출력하기 
    with training_args.main_process_first(
        desc="Log a few random samples from the processed training set"
    ):
        for index in random.sample(range(len(train_dataset)), 2):
            print(train_dataset[index]["text"])

    # Model 및 파라미터 설정하기 
    model = AutoModelForCausalLM.from_pretrained(
        script_args.model_name,
        attn_implementation="sdpa", 
        torch_dtype=torch.bfloat16,
        use_cache=False if training_args.gradient_checkpointing else True,  
    )
    
    if training_args.gradient_checkpointing:
        model.gradient_checkpointing_enable()

    # Train 설정 
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        dataset_text_field="text",
        eval_dataset=test_dataset,
        max_seq_length=script_args.max_seq_length,
        tokenizer=tokenizer,
        packing=True,
        dataset_kwargs={
            "add_special_tokens": False,  
            "append_concat_token": False, 
        },
    )

    checkpoint = None
    if training_args.resume_from_checkpoint is not None:
        checkpoint = training_args.resume_from_checkpoint
    trainer.train(resume_from_checkpoint=checkpoint)

    if trainer.is_fsdp_enabled:
        trainer.accelerator.state.fsdp_plugin.set_state_dict_type("FULL_STATE_DICT")
    trainer.save_model()

/Users/yoon/Desktop/GeorgiaTech-Assingment/Mental-Health-AI-Driven-System-Project/.venv/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


usage: ipykernel_launcher.py [-h] [--dataset_path DATASET_PATH]
                             [--model_name MODEL_NAME]
                             [--max_seq_length MAX_SEQ_LENGTH]
                             [--question_key QUESTION_KEY]
                             [--answer_key ANSWER_KEY] --output_dir OUTPUT_DIR
                             [--overwrite_output_dir [OVERWRITE_OUTPUT_DIR]]
                             [--do_train [DO_TRAIN]] [--do_eval [DO_EVAL]]
                             [--do_predict [DO_PREDICT]]
                             [--eval_strategy {no,steps,epoch}]
                             [--prediction_loss_only [PREDICTION_LOSS_ONLY]]
                             [--per_device_train_batch_size PER_DEVICE_TRAIN_BATCH_SIZE]
                             [--per_device_eval_batch_size PER_DEVICE_EVAL_BATCH_SIZE]
                             [--per_gpu_train_batch_size PER_GPU_TRAIN_BATCH_SIZE]
                             [--per_gpu_eval_batch_size PER_GPU_EVAL_BA

SystemExit: 2

/Users/yoon/Desktop/GeorgiaTech-Assingment/Mental-Health-AI-Driven-System-Project/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3557: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
